# 09102024

## 1. The basic structure of export process

* Aim: export fsusie generated results for ATAC-seq data in ROSMAP(single-cell) across 6 cell types and for mQTL data in Knight. In a bigger picture, after analyzing seq data, we want to generate readable results that can provide insight of biological problems. The following steps are:
    * 1. export ATAC-seq and mQTL data to show genome-wide significant QTLs or ATAC regions
    * 2. GWS export(skip): export genes or region lists that is associated with AD (considered as causal?)
    * 3. overlap: fild the overlap between 1 and 2 in hope of finding causal QTLs for AD.

* Input: s3://statfungen/ftp_fgc_xqtl/interactive_analysis/hs3163/knight_snATAC_fsusie/rds/ 
* Output: s3://statfungen/ftp_fgc_xqtl/analysis_result/finemapping_twas/fsusie_export/
* Process: readable tables and side products(output if exists) + QC + metadata.tsv(contain all info across cohort and contexts for one gene)

## 2. What I should focus on interms of learning
* sos advanced usage in pipeline fine_mapping_post_processing.ipynb

In [ ]:
if len(region_name) > 0:
    region_ids = list(set(region_ids).union(set(region_name)))

**i. getting list from two resourses**
The operation list(set(region_ids).union(set(region_name))) will not result in duplicates, even if region_ids and region_name contain the same strings. Let me explain why:
set() function: When you convert a list (or any iterable) to a set using set(), it automatically removes all duplicates. Sets in Python only contain unique elements.
union() method: The union() method of sets combines two sets, keeping only unique elements. It's equivalent to the mathematical union of sets.
Converting back to a list: After the union operation, converting the resulting set back to a list with list() will give you a list of unique elements.


In [ ]:
**ii.to skip the candidates that has been processed:**

In [ ]:
results_set = set(results['original_data']) results_set = {item.strip() for sub in results_set for item in sub.split(',')}

This creates a set of unique items from the 'original_data' column. It assumes 'original_data' contains comma-separated values, which are split and stripped of whitespace.


In [ ]:
mask = region['original_data'].apply(lambda paths: not is_file_exported(paths, results_set)) region = region[mask]


This creates a boolean mask by applying a function is_file_exported to each value in the 'original_data' column of the region DataFrame. The not operator inverts the result, so True becomes False and vice versa. The region DataFrame is then filtered to keep only the rows where the mask is True.


**iii.if else statement in sos pipeline**

In [ ]:
if(${"TRUE" if condition_meta.is_file() else "FALSE"}){
meta <- suppressMessages(read_delim("${condition_meta}", col_names = F))
context <- meta %>% filter(X1 == condition) %>% pull(X2)
if (length(context) == 0) {
context <- condition
message("No matching entries found. context has been set to the condition value.")
}
} else {context <- condition}


if(${"TRUE" if multi_gene else "FALSE"}) conditions <-dat_study[['condition_names']] %>% paste(., collapse = '_') else conditions <- names(dat_study)[sapply(dat_study, function(x) length(x) > 0)]



**3.get familiar with lab's data

## 3. How to customize fsusie data export commands


In [ ]:
sos run xqtl-protocol/pipeline/fine_mapping_post_processing.ipynb  cis_results_export    
--region_file  xqtl-analysis/resource/extended_TADB.bed  # reference 'gene_list'(can be region-list, block-list),have last column being region name.  
--file_path analysis_result/finemapping_twas/fsusie/     # # the path stored original output files, assuming orinal files are named as prefix.id.suffix, no change
--name Fungen_xQTL   # A region list file documenting the chr_pos_ref_alt of a susie_object,change to avoid overwritten
--suffix fsusie_mixture_normal_top_pc_weights.rds    # suffix of the original files (after id) to identify file.
--prefix MSBB_mQTL ROSMAP_mQTL ROSMAP_haQTL    # prefix of the original files (before id) to identify file, this is a list, indicating it can have multiple contexts or cohorts
--data_type qtl     # remove it !!!
--min_corr 0.8  # keep it cuz the default value is 0.5
--geno_ref  analysis_result/geno_align/Fungen_xQTL.ROSMAP_NIA_WGS.ROSMAP_NIA_WGS.MSBB_WGS_ADSP_hg38.MiGA.MAP_Brain-xQTL_Gwas_geno_0.STARNET.aligned.bim.gz  # the (aligned) geno bim file to check allele flipping
--context-meta  xqtl-analysis/resource/context_meta.tsv    # An index, need to add 7 rows(6 for ATAC-seq for 6 contexts; 1 for Knight for 1 context)
--cwd ../output 
--fsusie  # Keep it as the fsusie output format is different from that for susie and meta_susie
--region-name  chr1_0_7520074 # the option for mmjobman submission, blank for example.sh

## 4. NB
Don't forget to test before submitting a job(2 lines)